# Traveling salesperson (4 cities)

A salesperson leaves the **depot**, visits harbor, market and tower
once each, and returns. City 0 is pinned at time 0, so the remaining
$3\times 3$ one-hot matrix is 9 qubits.

The cost of a valid tour is its closed length. Invalid (not one-hot)
bitstrings get a flat penalty. The ansatz is a hardware-efficient
RY–CZ ladder — **not** QAOA — so this notebook stays independent of
`hybrid/qaoa`.

In [1]:
import itertools
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize

NAMES = ("depot", "harbor", "market", "tower")
DIST = np.array(
    [
        [0.0, 2.0, 3.0, 2.5],
        [2.0, 0.0, 1.5, 4.0],
        [3.0, 1.5, 0.0, 1.0],
        [2.5, 4.0, 1.0, 0.0],
    ]
)
N_FREE = 3
N = N_FREE * N_FREE
PENALTY = 8.0
LAYERS = 2


def idx(city, slot):
    return (city - 1) * N_FREE + (slot - 1)


def length(order):
    return sum(DIST[a, b] for a, b in zip(order, order[1:] + order[:1]))

## Classical baseline: every permutation of the three free cities

In [2]:
best, best_len = None, float("inf")
for tail in itertools.permutations((1, 2, 3)):
    tour = (0, *tail)
    L = length(tour)
    print(tour, [NAMES[i] for i in tour], L)
    if L < best_len:
        best, best_len = tour, L
print("optimum", best, best_len)

(0, 1, 2, 3) ['depot', 'harbor', 'market', 'tower'] 7.0
(0, 1, 3, 2) ['depot', 'harbor', 'tower', 'market'] 10.0
(0, 2, 1, 3) ['depot', 'market', 'harbor', 'tower'] 11.0
(0, 2, 3, 1) ['depot', 'market', 'tower', 'harbor'] 10.0
(0, 3, 1, 2) ['depot', 'tower', 'harbor', 'market'] 11.0
(0, 3, 2, 1) ['depot', 'tower', 'market', 'harbor'] 7.0
optimum (0, 1, 2, 3) 7.0


## One-hot decoder and QUBO energy

In [3]:
def decode(bits):
    flags = [int(b) for b in bits[::-1]]
    slots = [0, -1, -1, -1]
    used_c, used_t = set(), set()
    for city in (1, 2, 3):
        ones = [t for t in (1, 2, 3) if flags[idx(city, t)] == 1]
        if len(ones) != 1:
            return None
        t = ones[0]
        if t in used_t:
            return None
        slots[t] = city
        used_t.add(t)
        used_c.add(city)
    return tuple(slots)


def energy(bits):
    tour = decode(bits)
    return PENALTY if tour is None else length(tour)

## Hardware-efficient ansatz and a short COBYLA loop

In [4]:
def ansatz(theta):
    angles = theta.reshape(LAYERS + 1, N)
    qc = QuantumCircuit(N)
    for layer in range(LAYERS):
        for q in range(N):
            qc.ry(float(angles[layer, q]), q)
        for q in range(N - 1):
            qc.cz(q, q + 1)
    for q in range(N):
        qc.ry(float(angles[LAYERS, q]), q)
    return qc


def expected(theta):
    probs = Statevector.from_instruction(ansatz(theta)).probabilities_dict()
    return sum(p * energy(b) for b, p in probs.items())


rng = np.random.default_rng(21)
opt = minimize(
    expected,
    rng.normal(0, 0.4, size=(LAYERS + 1) * N),
    method="COBYLA",
    options={"maxiter": 60, "rhobeg": 0.5},
)
probs = Statevector.from_instruction(ansatz(opt.x)).probabilities_dict()
bits, p = max(probs.items(), key=lambda kv: kv[1])
tour = decode(bits)
print("expected energy", opt.fun)
print("most likely", bits, "P", round(p, 3), "tour", tour)
if tour:
    print("names", [NAMES[i] for i in tour], "length", length(tour))

expected energy 7.453992111857934
most likely 100010001 P 0.553 tour (0, 1, 2, 3)
names ['depot', 'harbor', 'market', 'tower'] length 7.0
